# A01. AI가입일 = 전자금융가입일 정합 + 이벤트 전후 비교

- 입력: 고객정보 TSV/CSV, (선택) 이벤트 파일
- 출력: 동일일 가입 비율, 이벤트 전후 지표
- 데이터 파일이 탭(`\t`) 구분자여도 그대로 읽히도록 구성

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

root = Path.cwd()
refactor_dir = root if root.name == 'refactor' else (root / 'analysis' / 'refactor')
if not refactor_dir.exists():
    refactor_dir = Path('/workspace/analysis/refactor')
if str(refactor_dir) not in sys.path:
    sys.path.insert(0, str(refactor_dir))

import common
import a01_signup_alignment as a01

print('refactor_dir =', refactor_dir)

In [ ]:
# 여기만 너 파일명으로 바꿔서 실행
PROFILE_FILE = '/home/jovyan/cjs/refactor2/data/user.csv'
EVENT_FILE = None  # 예: '/home/jovyan/cjs/refactor2/data/event.tsv'

profile = common.normalize_profile(common.load_csv(PROFILE_FILE))
events = common.normalize_event(common.load_csv(EVENT_FILE)) if EVENT_FILE else pd.DataFrame()

print('profile:', profile.shape)
print('events:', events.shape)
profile.head(2)

In [ ]:
alignment = a01.analyze_signup_alignment(profile)
daily = a01.build_daily_from_profile(profile)
event_table = a01.analyze_event_windows(daily, events) if not events.empty else pd.DataFrame()

display(alignment)
display(event_table)
print('해석:', a01.suggest_comment(alignment))

if not daily.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    sns.lineplot(data=daily, x='date', y='new_signups', marker='o', ax=axes[0], label='신규가입')
    sns.lineplot(data=daily, x='date', y='ai_signups', marker='o', ax=axes[0], label='AI가입')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].set_title('일자별 신규/AI가입')

    sns.lineplot(data=daily, x='date', y='same_day_conv_rate', marker='o', color='#dd8452', ax=axes[1])
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].set_title('동일일 전환율')
    axes[1].set_ylim(bottom=0)

    plt.tight_layout()
    plt.show()